In [22]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scipy.stats as stats

import nibabel as nib
import nilearn.connectome as nic
import nilearn.plotting as plotting
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import hcp_utils as hcp
from collections import Counter



from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, precision_recall_curve, balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Input, Concatenate, BatchNormalization, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Model



import tensorflow as tf
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.combine import SMOTETomek

from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests
from scipy.stats import pearsonr

## d15

In [23]:
from nilearn import image, plotting
from atlasreader.atlasreader import read_atlas_peak

"""
available reference atlases
---------------------------
    "aal",
    "aicha",
    "desikan_killiany",
    "destrieux",
    "harvard_oxford",
    "juelich",
    "marsatlas",
    "neuromorphometrics",
    "talairach_ba",
    "talairach_gyrus",
 """

atlas = image.threshold_img("HCP_PTN1200/groupICA/groupICA_3T_HCP1200_MSMAll_d15.ica/melodic_IC_sum.nii.gz", "99.5%") 
atlas_coords = plotting.find_probabilistic_atlas_cut_coords(atlas)
brain_region = []
print("BRAIN REGIONS:\n--------------")
for atlas_coord in atlas_coords:
    region = read_atlas_peak("harvard_oxford", atlas_coord)
    print(region)
    brain_region += [region]

brain_region = [
    max(inner_list, key=lambda x: x[0])[-1] if inner_list else None for inner_list in brain_region
]

open_access_data = pd.read_csv("unrestricted_data.csv")
restricted_data = pd.read_csv("RESTRICTED_BEHAVIORAL_DATA.csv")
subject_data = open_access_data.merge(restricted_data, how = 'inner', on = 'Subject')

folder = 'HCP_PTN1200/node_timeseries/3T_HCP1200_MSMAll_d15_ts2'
brain_files = [f for f in os.listdir(folder) if f.endswith('.txt')]
brain_data = {}

for filename in brain_files:
    subject_id = int(filename[:6])
    file_path = os.path.join(folder, filename)
    subject_brain_data = np.loadtxt(file_path)
    brain_data[subject_id] = subject_brain_data

brain_data_df = pd.DataFrame({
    'Subject': list(brain_data.keys()),
    'Brain_Data': list(brain_data.values())  # (4800, 100) arrays
})
brain_data_df

data = subject_data.merge(brain_data_df, on='Subject', how='inner')

#Correlation matrix
correlation_measure = nic.ConnectivityMeasure(kind='partial correlation')
correlation_matrix = correlation_measure.fit_transform(data["Brain_Data"])
handedness = data["Handedness"]

#replace 1.0 with np.nan correlations
copy_matrix = correlation_matrix.copy()
copy_matrix[copy_matrix == 1.00] = np.nan

fisher_z_matrices = np.arctanh(copy_matrix)

#function to find significant correlations
def sig_pvals(fisher_z_matrices):
    corr_mat = {}
    pval_mat = {}
    count_pval = 0
    matrix_dim = fisher_z_matrices.shape
    for i in range(matrix_dim[1]):
        for j in range(i, matrix_dim[1]):
            if i != j:
                corr, pval = stats.pearsonr(fisher_z_matrices[:,i, j],  handedness)
                corr_mat[(i, j)] = corr
                pval_mat[(i,j)] = pval
                if abs(pval) < 0.05:
                    # print(f'pair ({i}, {j}) has a pval of {pval}')
                    count_pval += 1
    print(count_pval)
    return corr_mat, pval_mat
    
#find strongest negative correlation
corr_mat, pval_mat = sig_pvals(fisher_z_matrices)

sorted_keys = sorted(corr_mat, key=lambda x: corr_mat[x])

order = 0 # this corresponds to the sorted order, so 0 is the 1st values in the sorted keys
#you can use this to find top 10 regions, and bottom 10 regions

top_5_15 = sorted_keys[-5:]

bottom_5_15 = sorted_keys[:5]


BRAIN REGIONS:
--------------
[[47.0, 'Right_Occipital_Pole'], [23.0, 'Right_Lateral_Occipital_Cortex_superior_division']]
[[90.0, 'Left_Lateral_Occipital_Cortex_superior_division']]
[[73.0, 'Right_Occipital_Pole']]
[[58.0, 'Right_Lateral_Occipital_Cortex_superior_division']]
[[57.0, 'Left_Lateral_Occipital_Cortex_superior_division']]
[[75.0, 'Left_Supramarginal_Gyrus_anterior_division'], [8.0, 'Left_Supramarginal_Gyrus_posterior_division'], [6.0, 'Left_Parietal_Operculum_Cortex']]
[[56.0, 'Right_Angular_Gyrus'], [29.0, 'Right_Lateral_Occipital_Cortex_superior_division']]
[[8.0, 'Right_Occipital_Fusiform_Gyrus']]
[[77.0, 'Right_Lateral_Occipital_Cortex_superior_division']]
[[39.0, 'Left_Supramarginal_Gyrus_posterior_division'], [31.0, 'Left_Angular_Gyrus']]
[[37.0, 'Left_Postcentral_Gyrus'], [28.0, 'Left_Supramarginal_Gyrus_anterior_division'], [8.0, 'Left_Superior_Parietal_Lobule']]
[[26.0, 'Right_Supracalcarine_Cortex'], [23.0, 'Right_Intracalcarine_Cortex'], [13.0, 'Right_Lingual_Gy

In [24]:
print("Top 5 positively correlated regions:")
for region in top_5_15:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")

print("\nBottom 5 negatively correlated regions:")
for region in bottom_5_15:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")


Top 5 positively correlated regions:
Region pair (4, 10): Correlation 0.0872
Region pair (1, 9): Correlation 0.0975
Region pair (0, 7): Correlation 0.0995
Region pair (11, 13): Correlation 0.1006
Region pair (6, 14): Correlation 0.1168

Bottom 5 negatively correlated regions:
Region pair (1, 6): Correlation -0.1491
Region pair (13, 14): Correlation -0.1271
Region pair (6, 9): Correlation -0.1181
Region pair (4, 14): Correlation -0.0982
Region pair (1, 14): Correlation -0.0973


## d25

In [25]:
atlas = image.threshold_img("HCP_PTN1200/groupICA/groupICA_3T_HCP1200_MSMAll_d25.ica/melodic_IC_sum.nii.gz", "99.5%") 
atlas_coords = plotting.find_probabilistic_atlas_cut_coords(atlas)
brain_region = []
print("BRAIN REGIONS:\n--------------")
for atlas_coord in atlas_coords:
    region = read_atlas_peak("harvard_oxford", atlas_coord)
    print(region)
    brain_region += [region]

brain_region = [
    max(inner_list, key=lambda x: x[0])[-1] if inner_list else None for inner_list in brain_region
]

open_access_data = pd.read_csv("unrestricted_data.csv")
restricted_data = pd.read_csv("RESTRICTED_BEHAVIORAL_DATA.csv")
subject_data = open_access_data.merge(restricted_data, how = 'inner', on = 'Subject')

folder = 'HCP_PTN1200/node_timeseries/3T_HCP1200_MSMAll_d25_ts2'
brain_files = [f for f in os.listdir(folder) if f.endswith('.txt')]
brain_data = {}

for filename in brain_files:
    subject_id = int(filename[:6])
    file_path = os.path.join(folder, filename)
    subject_brain_data = np.loadtxt(file_path)
    brain_data[subject_id] = subject_brain_data

brain_data_df = pd.DataFrame({
    'Subject': list(brain_data.keys()),
    'Brain_Data': list(brain_data.values())  # (4800, 100) arrays
})
brain_data_df

data = subject_data.merge(brain_data_df, on='Subject', how='inner')

#Correlation matrix
correlation_measure = nic.ConnectivityMeasure(kind='partial correlation')
correlation_matrix = correlation_measure.fit_transform(data["Brain_Data"])
handedness = data["Handedness"]

#replace 1.0 with np.nan correlations
copy_matrix = correlation_matrix.copy()
copy_matrix[copy_matrix == 1.00] = np.nan

fisher_z_matrices = np.arctanh(copy_matrix)

#function to find significant correlations
def sig_pvals(fisher_z_matrices):
    corr_mat = {}
    pval_mat = {}
    count_pval = 0
    matrix_dim = fisher_z_matrices.shape
    for i in range(matrix_dim[1]):
        for j in range(i, matrix_dim[1]):
            if i != j:
                corr, pval = stats.pearsonr(fisher_z_matrices[:,i, j],  handedness)
                corr_mat[(i, j)] = corr
                pval_mat[(i,j)] = pval
                if abs(pval) < 0.05:
                    # print(f'pair ({i}, {j}) has a pval of {pval}')
                    count_pval += 1
    print(count_pval)
    return corr_mat, pval_mat
    
#find strongest negative correlation
corr_mat, pval_mat = sig_pvals(fisher_z_matrices)

sorted_keys = sorted(corr_mat, key=lambda x: corr_mat[x])

order = 0 # this corresponds to the sorted order, so 0 is the 1st values in the sorted keys
#you can use this to find top 10 regions, and bottom 10 regions

top_5_25 = sorted_keys[-5:]

bottom_5_25 = sorted_keys[:5]


BRAIN REGIONS:
--------------
[[71.0, 'Right_Occipital_Pole']]
[[82.0, 'Left_Lateral_Occipital_Cortex_superior_division']]
[[47.0, 'Right_Cuneal_Cortex'], [25.0, 'Left_Cuneal_Cortex']]
[[51.0, 'Right_Lateral_Occipital_Cortex_superior_division']]
[[29.0, 'Right_Lateral_Occipital_Cortex_superior_division'], [26.0, 'Right_Angular_Gyrus']]
[[44.0, 'Left_Angular_Gyrus'], [40.0, 'Left_Lateral_Occipital_Cortex_superior_division']]
[[64.0, 'Right_Occipital_Pole']]
[[32.0, 'Left_Precuneous_Cortex']]
[[40.0, 'Right_Supramarginal_Gyrus_anterior_division'], [20.0, 'Right_Parietal_Operculum_Cortex'], [11.0, 'Right_Planum_Temporale']]
[[43.0, 'Left_Postcentral_Gyrus'], [30.0, 'Left_Supramarginal_Gyrus_anterior_division']]
[[70.0, 'Left_Lateral_Occipital_Cortex_superior_division']]
[[51.0, 'Left_Supramarginal_Gyrus_anterior_division'], [29.0, 'Left_Supramarginal_Gyrus_posterior_division']]
[[49.0, 'Left_Postcentral_Gyrus'], [10.0, 'Left_Precentral_Gyrus']]
[[32.0, 'Left_Planum_Temporale'], [24.0, 'Le

In [26]:
print("Top 5 positively correlated regions:")
for region in top_5_25:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")

print("\nBottom 5 negatively correlated regions:")
for region in bottom_5_25:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")


Top 5 positively correlated regions:
Region pair (1, 5): Correlation 0.1072
Region pair (7, 20): Correlation 0.1115
Region pair (6, 7): Correlation 0.1140
Region pair (16, 20): Correlation 0.1167
Region pair (5, 17): Correlation 0.1342

Bottom 5 negatively correlated regions:
Region pair (17, 20): Correlation -0.2097
Region pair (1, 4): Correlation -0.1925
Region pair (15, 16): Correlation -0.1596
Region pair (4, 11): Correlation -0.1333
Region pair (16, 17): Correlation -0.1234


## d50

In [27]:
atlas = image.threshold_img("HCP_PTN1200/groupICA/groupICA_3T_HCP1200_MSMAll_d50.ica/melodic_IC_sum.nii.gz", "99.5%") 
atlas_coords = plotting.find_probabilistic_atlas_cut_coords(atlas)
brain_region = []
for atlas_coord in atlas_coords:
    region = read_atlas_peak("harvard_oxford", atlas_coord)
    brain_region += [region]

brain_region = [
    max(inner_list, key=lambda x: x[0])[-1] if inner_list else None for inner_list in brain_region
]

open_access_data = pd.read_csv("unrestricted_data.csv")
restricted_data = pd.read_csv("RESTRICTED_BEHAVIORAL_DATA.csv")
subject_data = open_access_data.merge(restricted_data, how = 'inner', on = 'Subject')

folder = 'HCP_PTN1200/node_timeseries/3T_HCP1200_MSMAll_d50_ts2'
brain_files = [f for f in os.listdir(folder) if f.endswith('.txt')]
brain_data = {}

for filename in brain_files:
    subject_id = int(filename[:6])
    file_path = os.path.join(folder, filename)
    subject_brain_data = np.loadtxt(file_path)
    brain_data[subject_id] = subject_brain_data

brain_data_df = pd.DataFrame({
    'Subject': list(brain_data.keys()),
    'Brain_Data': list(brain_data.values())  # (4800, 100) arrays
})
brain_data_df

data = subject_data.merge(brain_data_df, on='Subject', how='inner')

#Correlation matrix
correlation_measure = nic.ConnectivityMeasure(kind='partial correlation')
correlation_matrix = correlation_measure.fit_transform(data["Brain_Data"])
handedness = data["Handedness"]

#replace 1.0 with np.nan correlations
copy_matrix = correlation_matrix.copy()
copy_matrix[copy_matrix == 1.00] = np.nan

fisher_z_matrices = np.arctanh(copy_matrix)

#function to find significant correlations
def sig_pvals(fisher_z_matrices):
    corr_mat = {}
    pval_mat = {}
    count_pval = 0
    matrix_dim = fisher_z_matrices.shape
    for i in range(matrix_dim[1]):
        for j in range(i, matrix_dim[1]):
            if i != j:
                corr, pval = stats.pearsonr(fisher_z_matrices[:,i, j],  handedness)
                corr_mat[(i, j)] = corr
                pval_mat[(i,j)] = pval
                if abs(pval) < 0.05:
                    # print(f'pair ({i}, {j}) has a pval of {pval}')
                    count_pval += 1
    print(count_pval)
    return corr_mat, pval_mat
    
#find strongest negative correlation
corr_mat, pval_mat = sig_pvals(fisher_z_matrices)

sorted_keys = sorted(corr_mat, key=lambda x: corr_mat[x])

order = 0 # this corresponds to the sorted order, so 0 is the 1st values in the sorted keys
#you can use this to find top 10 regions, and bottom 10 regions

top_5_50 = sorted_keys[-5:]

bottom_5_50 = sorted_keys[:5]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [28]:
print("Top 5 positively correlated regions:")
for region in top_5_50:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")

print("\nBottom 5 negatively correlated regions:")
for region in bottom_5_50:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")


Top 5 positively correlated regions:
Region pair (14, 25): Correlation 0.1093
Region pair (6, 16): Correlation 0.1167
Region pair (4, 43): Correlation 0.1420
Region pair (4, 11): Correlation 0.1609
Region pair (8, 29): Correlation 0.2212

Bottom 5 negatively correlated regions:
Region pair (8, 14): Correlation -0.1875
Region pair (6, 29): Correlation -0.1850
Region pair (11, 27): Correlation -0.1790
Region pair (25, 29): Correlation -0.1744
Region pair (6, 22): Correlation -0.1349


## d100

In [29]:
atlas = image.threshold_img("HCP_PTN1200/groupICA/groupICA_3T_HCP1200_MSMAll_d100.ica/melodic_IC_sum.nii.gz", "99.5%") 
atlas_coords = plotting.find_probabilistic_atlas_cut_coords(atlas)
brain_region = []
for atlas_coord in atlas_coords:
    region = read_atlas_peak("harvard_oxford", atlas_coord)
    brain_region += [region]

brain_region = [
    max(inner_list, key=lambda x: x[0])[-1] if inner_list else None for inner_list in brain_region
]

open_access_data = pd.read_csv("unrestricted_data.csv")
restricted_data = pd.read_csv("RESTRICTED_BEHAVIORAL_DATA.csv")
subject_data = open_access_data.merge(restricted_data, how = 'inner', on = 'Subject')

folder = 'HCP_PTN1200/node_timeseries/3T_HCP1200_MSMAll_d100_ts2'
brain_files = [f for f in os.listdir(folder) if f.endswith('.txt')]
brain_data = {}

for filename in brain_files:
    subject_id = int(filename[:6])
    file_path = os.path.join(folder, filename)
    subject_brain_data = np.loadtxt(file_path)
    brain_data[subject_id] = subject_brain_data

brain_data_df = pd.DataFrame({
    'Subject': list(brain_data.keys()),
    'Brain_Data': list(brain_data.values())  # (4800, 100) arrays
})
brain_data_df

data = subject_data.merge(brain_data_df, on='Subject', how='inner')

#Correlation matrix
correlation_measure = nic.ConnectivityMeasure(kind='partial correlation')
correlation_matrix = correlation_measure.fit_transform(data["Brain_Data"])
handedness = data["Handedness"]

#replace 1.0 with np.nan correlations
copy_matrix = correlation_matrix.copy()
copy_matrix[copy_matrix == 1.00] = np.nan

fisher_z_matrices = np.arctanh(copy_matrix)

#function to find significant correlations
def sig_pvals(fisher_z_matrices):
    corr_mat = {}
    pval_mat = {}
    count_pval = 0
    matrix_dim = fisher_z_matrices.shape
    for i in range(matrix_dim[1]):
        for j in range(i, matrix_dim[1]):
            if i != j:
                corr, pval = stats.pearsonr(fisher_z_matrices[:,i, j],  handedness)
                corr_mat[(i, j)] = corr
                pval_mat[(i,j)] = pval
                if abs(pval) < 0.05:
                    # print(f'pair ({i}, {j}) has a pval of {pval}')
                    count_pval += 1
    print(count_pval)
    return corr_mat, pval_mat
    
#find strongest negative correlation
corr_mat, pval_mat = sig_pvals(fisher_z_matrices)

sorted_keys = sorted(corr_mat, key=lambda x: corr_mat[x])

order = 0 # this corresponds to the sorted order, so 0 is the 1st values in the sorted keys
#you can use this to find top 10 regions, and bottom 10 regions

top_5_100 = sorted_keys[-5:]

bottom_5_100 = sorted_keys[:5]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [30]:
print("Top 5 positively correlated regions:")
for region in top_5_100:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")

print("\nBottom 5 negatively correlated regions:")
for region in bottom_5_100:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")


Top 5 positively correlated regions:
Region pair (9, 30): Correlation 0.1247
Region pair (8, 17): Correlation 0.1322
Region pair (8, 26): Correlation 0.1326
Region pair (13, 43): Correlation 0.1512
Region pair (4, 30): Correlation 0.2187

Bottom 5 negatively correlated regions:
Region pair (30, 41): Correlation -0.1430
Region pair (4, 36): Correlation -0.1419
Region pair (6, 13): Correlation -0.1409
Region pair (11, 32): Correlation -0.1407
Region pair (7, 35): Correlation -0.1348


## d200

In [31]:
atlas = image.threshold_img("HCP_PTN1200/groupICA/groupICA_3T_HCP1200_MSMAll_d200.ica/melodic_IC_sum.nii.gz", "99.5%") 
atlas_coords = plotting.find_probabilistic_atlas_cut_coords(atlas)
brain_region = []
for atlas_coord in atlas_coords:
    region = read_atlas_peak("harvard_oxford", atlas_coord)
    brain_region += [region]

brain_region = [
    max(inner_list, key=lambda x: x[0])[-1] if inner_list else None for inner_list in brain_region
]

open_access_data = pd.read_csv("unrestricted_data.csv")
restricted_data = pd.read_csv("RESTRICTED_BEHAVIORAL_DATA.csv")
subject_data = open_access_data.merge(restricted_data, how = 'inner', on = 'Subject')

folder = 'HCP_PTN1200/node_timeseries/3T_HCP1200_MSMAll_d200_ts2'
brain_files = [f for f in os.listdir(folder) if f.endswith('.txt')]
brain_data = {}

for filename in brain_files:
    subject_id = int(filename[:6])
    file_path = os.path.join(folder, filename)
    subject_brain_data = np.loadtxt(file_path)
    brain_data[subject_id] = subject_brain_data

brain_data_df = pd.DataFrame({
    'Subject': list(brain_data.keys()),
    'Brain_Data': list(brain_data.values())  # (4800, 100) arrays
})
brain_data_df

data = subject_data.merge(brain_data_df, on='Subject', how='inner')

#Correlation matrix
correlation_measure = nic.ConnectivityMeasure(kind='partial correlation')
correlation_matrix = correlation_measure.fit_transform(data["Brain_Data"])
handedness = data["Handedness"]

#replace 1.0 with np.nan correlations
copy_matrix = correlation_matrix.copy()
copy_matrix[copy_matrix == 1.00] = np.nan

fisher_z_matrices = np.arctanh(copy_matrix)

#function to find significant correlations
def sig_pvals(fisher_z_matrices):
    corr_mat = {}
    pval_mat = {}
    count_pval = 0
    matrix_dim = fisher_z_matrices.shape
    for i in range(matrix_dim[1]):
        for j in range(i, matrix_dim[1]):
            if i != j:
                corr, pval = stats.pearsonr(fisher_z_matrices[:,i, j],  handedness)
                corr_mat[(i, j)] = corr
                pval_mat[(i,j)] = pval
                if abs(pval) < 0.05:
                    # print(f'pair ({i}, {j}) has a pval of {pval}')
                    count_pval += 1
    print(count_pval)
    return corr_mat, pval_mat
    
#find strongest negative correlation
corr_mat, pval_mat = sig_pvals(fisher_z_matrices)

sorted_keys = sorted(corr_mat, key=lambda x: corr_mat[x])

order = 0 # this corresponds to the sorted order, so 0 is the 1st values in the sorted keys
#you can use this to find top 10 regions, and bottom 10 regions

top_5_200 = sorted_keys[-5:]

bottom_5_200 = sorted_keys[:5]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [32]:
print("Top 5 positively correlated regions:")
for region in top_5_200:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")

print("\nBottom 5 negatively correlated regions:")
for region in bottom_5_200:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")


Top 5 positively correlated regions:
Region pair (15, 49): Correlation 0.1323
Region pair (12, 51): Correlation 0.1397
Region pair (51, 58): Correlation 0.1442
Region pair (4, 26): Correlation 0.1548
Region pair (33, 133): Correlation 0.1860

Bottom 5 negatively correlated regions:
Region pair (9, 18): Correlation -0.1607
Region pair (5, 51): Correlation -0.1448
Region pair (38, 51): Correlation -0.1383
Region pair (33, 155): Correlation -0.1286
Region pair (3, 15): Correlation -0.1277


## d300

In [33]:
atlas = image.threshold_img("HCP_PTN1200/groupICA/groupICA_3T_HCP1200_MSMAll_d300.ica/melodic_IC_sum.nii.gz", "99.5%") 
atlas_coords = plotting.find_probabilistic_atlas_cut_coords(atlas)
brain_region = []
for atlas_coord in atlas_coords:
    region = read_atlas_peak("harvard_oxford", atlas_coord)
    brain_region += [region]

brain_region = [
    max(inner_list, key=lambda x: x[0])[-1] if inner_list else None for inner_list in brain_region
]

open_access_data = pd.read_csv("unrestricted_data.csv")
restricted_data = pd.read_csv("RESTRICTED_BEHAVIORAL_DATA.csv")
subject_data = open_access_data.merge(restricted_data, how = 'inner', on = 'Subject')

folder = 'HCP_PTN1200/node_timeseries/3T_HCP1200_MSMAll_d300_ts2'
brain_files = [f for f in os.listdir(folder) if f.endswith('.txt')]
brain_data = {}

for filename in brain_files:
    subject_id = int(filename[:6])
    file_path = os.path.join(folder, filename)
    subject_brain_data = np.loadtxt(file_path)
    brain_data[subject_id] = subject_brain_data

brain_data_df = pd.DataFrame({
    'Subject': list(brain_data.keys()),
    'Brain_Data': list(brain_data.values())  # (4800, 100) arrays
})
brain_data_df

data = subject_data.merge(brain_data_df, on='Subject', how='inner')

#Correlation matrix
correlation_measure = nic.ConnectivityMeasure(kind='partial correlation')
correlation_matrix = correlation_measure.fit_transform(data["Brain_Data"])
handedness = data["Handedness"]

#replace 1.0 with np.nan correlations
copy_matrix = correlation_matrix.copy()
copy_matrix[copy_matrix == 1.00] = np.nan

fisher_z_matrices = np.arctanh(copy_matrix)

#function to find significant correlations
def sig_pvals(fisher_z_matrices):
    corr_mat = {}
    pval_mat = {}
    count_pval = 0
    matrix_dim = fisher_z_matrices.shape
    for i in range(matrix_dim[1]):
        for j in range(i, matrix_dim[1]):
            if i != j:
                corr, pval = stats.pearsonr(fisher_z_matrices[:,i, j],  handedness)
                corr_mat[(i, j)] = corr
                pval_mat[(i,j)] = pval
                if abs(pval) < 0.05:
                    # print(f'pair ({i}, {j}) has a pval of {pval}')
                    count_pval += 1
    print(count_pval)
    return corr_mat, pval_mat
    
#find strongest negative correlation
corr_mat, pval_mat = sig_pvals(fisher_z_matrices)

sorted_keys = sorted(corr_mat, key=lambda x: corr_mat[x])

order = 0 # this corresponds to the sorted order, so 0 is the 1st values in the sorted keys
#you can use this to find top 10 regions, and bottom 10 regions

top_5_300 = sorted_keys[-5:]

bottom_5_300 = sorted_keys[:5]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [34]:
print("Top 5 positively correlated regions:")
for region in top_5_300:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")

print("\nBottom 5 negatively correlated regions:")
for region in bottom_5_300:
    print(f"Region pair {region}: Correlation {corr_mat[region]:.4f}")


Top 5 positively correlated regions:
Region pair (126, 131): Correlation 0.1435
Region pair (38, 43): Correlation 0.1449
Region pair (30, 45): Correlation 0.1463
Region pair (23, 78): Correlation 0.1654
Region pair (30, 60): Correlation 0.1905

Bottom 5 negatively correlated regions:
Region pair (23, 63): Correlation -0.2127
Region pair (4, 23): Correlation -0.1691
Region pair (24, 38): Correlation -0.1614
Region pair (55, 72): Correlation -0.1538
Region pair (24, 60): Correlation -0.1484
